# Phase 9 — Evaluation: 10 test claims

Runs every scenario in `backend/generate_synthetic_data.py`'s `SCENARIOS` list through
the live orchestrator graph (`AGENT_MODE=orchestrator`, the current default —
`backend/agent/orchestrator.py`) and diffs the actual decision against the predetermined
expected outcome in [`specs/eval_claims.md`](../specs/eval_claims.md).

Each run is a **real** claim: a fresh row in `claims`/`check_ledger`/`audit_trail`,
driven through `graph.invoke(...)` exactly like `backend/smoke_test_orchestrator.py`,
against the real Postgres dev DB, real Qdrant collection, and real LLM (`gpt-5.6-luna`
by default, per `.env.local`'s `LLM_PROVIDER`/`OPENAI_MODEL`). Nothing here is mocked.
Test claims are deleted at the end of the notebook so repeated runs don't accumulate
junk rows in the dev DB.

In [1]:
import os
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's own directory (backend/); the
# project's packages are imported as `backend.*`, which requires the repo root (the
# parent of backend/) on sys.path instead.
repo_root = Path.cwd()
if repo_root.name == "backend":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)
print("repo root:", repo_root)

repo root: /Users/paritosh.mathur/projects/agentic-claim-processing


In [2]:
import json
from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv

load_dotenv(".env.local")

from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.types import Command

from backend import db
from backend.agent.graph import initial_state
from backend.agent.orchestrator import build_orchestrator_graph
from backend.generate_synthetic_data import SCENARIOS

print(f"{len(SCENARIOS)} scenarios loaded from generate_synthetic_data.SCENARIOS")

10 scenarios loaded from generate_synthetic_data.SCENARIOS


## Expected outcomes

Predetermined **before** looking at any run's actual result — see
`specs/eval_claims.md` for the full table and the reasoning behind each expected
decision. `HUMAN_ANSWERS` is this notebook's auto-responder for the three scenarios
(#6/#7/#10) whose `account_profiles` row is deliberately not loaded, forcing
`ask_human`; every other scenario resolves purely from tool calls and never triggers
it.

In [3]:
EXPECTED = {
    "ACC-9001": "approve",
    "ACC-9002": "approve",
    "ACC-9003": "approve",
    "ACC-9004": "deny",
    "ACC-9005": "deny",
    "ACC-9006": "approve",
    "ACC-9007": "deny",
    "ACC-9008": "deny",
    "ACC-9009": "deny",
    "ACC-9010": "inconclusive",
}

HUMAN_ANSWERS = {
    "ACC-9006": "yes",
    "ACC-9007": "no",
    "ACC-9010": "not sure, can't confirm either way",
}

expected_ids = set(EXPECTED)
scenario_ids = {s["account_id"] for s in SCENARIOS}
assert expected_ids == scenario_ids, f"EXPECTED out of sync with SCENARIOS: {expected_ids ^ scenario_ids}"


## Driver

Same pattern as `backend/smoke_test_orchestrator.py`'s `run_scenario`/`insert_claim`,
extended to (a) use a per-scenario human answer instead of always `"yes"`, and (b)
pull each check's full `detail` JSON, not just its status, so a mismatch can be
diagnosed without a second query.

In [4]:
def insert_claim(claim_id, claim_type, claim_payload):
    with db.get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO claims (id, claim_type, claim_payload, status) VALUES (%s, %s, %s, 'pending')",
                (claim_id, claim_type, json.dumps(claim_payload)),
            )


def fetch_result(claim_id):
    with db.get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT status, decision, decision_reason FROM claims WHERE id = %s", (claim_id,))
            claim_row = cur.fetchone()
            cur.execute(
                "SELECT check_name, check_status, detail FROM check_ledger WHERE claim_id = %s ORDER BY check_name",
                (claim_id,),
            )
            checks = {c["check_name"]: {"status": c["check_status"], "detail": c["detail"]} for c in cur.fetchall()}
    return {"status": claim_row["status"], "decision": claim_row["decision"], "reason": claim_row["decision_reason"], "checks": checks}


def run_scenario(scenario, max_human_turns=5):
    claim_id = str(uuid4())
    claim_type = scenario["claim_type"]
    claim_payload = scenario["claim_payload"]
    account_id = scenario["account_id"]
    answer = HUMAN_ANSWERS.get(account_id)

    insert_claim(claim_id, claim_type, claim_payload)
    with PostgresSaver.from_conn_string(db.DATABASE_URL) as checkpointer:
        graph = build_orchestrator_graph(checkpointer)
        config = {"configurable": {"thread_id": claim_id}}
        result = graph.invoke(initial_state(claim_id, claim_type, claim_payload), config=config)

        turns = 0
        while "__interrupt__" in result and turns < max_human_turns:
            interrupt_info = result["__interrupt__"][0]
            question = interrupt_info.value["question"]
            print(f"    [ask_human] {question!r} -> answering {answer!r}")
            if answer is None:
                raise RuntimeError(f"{account_id} hit ask_human but no HUMAN_ANSWERS entry is configured for it")
            result = graph.invoke(Command(resume=answer), config=config)
            turns += 1

    return claim_id, fetch_result(claim_id)


## Run all 10 claims through the live orchestrator

In [5]:
db.open_pool()

rows = []
for scenario in SCENARIOS:
    account_id = scenario["account_id"]
    print(f"=== {account_id} ({scenario['claim_type']} / {scenario['claim_payload']['reason']}) ===")
    claim_id, result = run_scenario(scenario)
    expected = EXPECTED[account_id]
    actual = result["decision"]
    match = actual == expected
    print(f"  expected={expected!r} actual={actual!r} match={match} reason={result['reason']!r}\n")
    rows.append({
        "account_id": account_id,
        "claim_id": claim_id,
        "claim_type": scenario["claim_type"],
        "reason": scenario["claim_payload"]["reason"],
        "expected": expected,
        "actual": actual,
        "match": match,
        "decision_reason": result["reason"],
        "checks": result["checks"],
    })


=== ACC-9001 (fraud / unauthorized_transaction) ===


  expected='approve' actual='approve' match=True reason='All required checks passed'

=== ACC-9002 (billing_dispute / duplicate_charge) ===


  expected='approve' actual='approve' match=True reason='All required checks passed'

=== ACC-9003 (fraud / unauthorized_transaction) ===


  expected='approve' actual='approve' match=True reason='All required checks passed'

=== ACC-9004 (fraud / unauthorized_transaction) ===


  expected='deny' actual='deny' match=True reason='Required check(s) failed: account_red_flags'

=== ACC-9005 (billing_dispute / duplicate_charge) ===


...

    [ask_human] 'Is account ACC-9007 in good standing (not restricted, delinquent, or otherwise subject to adverse account-status flags)?' -> answering 'no'
  expected='deny' actual='deny' match=True reason='Required check(s) failed: account_standing'

=== ACC-9008 (billing_dispute / not_recognized) ===


  expected='deny' actual='deny' match=True reason='Required check(s) failed: transaction_exists'

=== ACC-9009 (fraud / unauthorized_transaction) ===


  expected='deny' actual='deny' match=True reason='Required check(s) failed: system_access_log_check'

=== ACC-9010 (fraud / other) ===


    [ask_human] 'Does account ACC-9010 have any documented account-level red flags (such as prior confirmed fraud, an active investigation, or a restriction) relevant to this claim?' -> answering "not sure, can't confirm either way"


  expected='inconclusive' actual='inconclusive' match=True reason='Required check(s) unresolved: account_red_flags'



## Results

In [6]:
df = pd.DataFrame(rows)[["account_id", "claim_type", "reason", "expected", "actual", "match", "decision_reason"]]
df


,account_id,claim_type,reason,expected,actual,match,decision_reason
0,ACC-9001,fraud,unauthorized_transaction,approve,approve,True,All required checks passed
1,ACC-9002,billing_dispute,duplicate_charge,approve,approve,True,All required checks passed
2,ACC-9003,fraud,unauthorized_transaction,approve,approve,True,All required checks passed
3,ACC-9004,fraud,unauthorized_transaction,deny,deny,True,Required check(s) failed: account_red_flags
4,ACC-9005,billing_dispute,duplicate_charge,deny,deny,True,Required check(s) failed: duplicate_charge_check
5,ACC-9006,fraud,not_recognized,approve,approve,True,All required checks passed
6,ACC-9007,billing_dispute,duplicate_charge,deny,deny,True,Required check(s) failed: account_standing
7,ACC-9008,billing_dispute,not_recognized,deny,deny,True,Required check(s) failed: transaction_exists
8,ACC-9009,fraud,unauthorized_transaction,deny,deny,True,Required check(s) failed: system_access_log_check
9,ACC-9010,fraud,other,inconclusive,inconclusive,True,Required check(s) unresolved: account_red_flags


In [7]:
n_match = sum(r["match"] for r in rows)
print(f"{n_match}/{len(rows)} claims matched their expected decision.")

if n_match < len(rows):
    print("\nMismatches (full check-ledger detail):")
    for r in rows:
        if not r["match"]:
            print(f"\n{r['account_id']}: expected {r['expected']!r}, got {r['actual']!r}")
            print(f"  decision_reason: {r['decision_reason']!r}")
            for name, c in r["checks"].items():
                print(f"    {name}: {c['status']}  {json.dumps(c['detail'])[:200]}")

assert n_match == len(rows), f"{len(rows) - n_match} claim(s) did not match their expected outcome -- see mismatches above"
print("\nAll claims matched. Eval set passes.")


10/10 claims matched their expected decision.

All claims matched. Eval set passes.


## Cleanup

Each run creates 10 fresh claims (`uuid4()` claim ids). Delete them so re-running this
notebook doesn't leave a pile of eval claims in the dev DB — `check_ledger`/`audit_trail`
rows cascade-delete with them (`schema.sql`'s `ON DELETE CASCADE`). The synthetic
`account_profiles`/`transactions`/`access_logs` fixtures are untouched (those are
long-lived scenario data, loaded once by `generate_synthetic_data.py`, not per-run).

In [8]:
with db.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("DELETE FROM claims WHERE id = ANY(%s)", ([r["claim_id"] for r in rows],))
db.close_pool()
print(f"Cleaned up {len(rows)} eval claims from claims/check_ledger/audit_trail.")


Cleaned up 10 eval claims from claims/check_ledger/audit_trail.
